# Blackberry 3DGS on Google Colab (`engines_cuda`)

Uses **PyTorch + gsplat** (CUDA). Aims at a Postshot-like recipe via **MCMC** densify
(`--strategy mcmc --cap-max 200000`), matching `danny_splat_45000.ply` scale.

**Runtime → GPU** required. Clone branch: `feature/cuda-torch-train`.

PyTorch3D is optional (meshes only) — see last section.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.__version__, torch.cuda.get_device_name(0), "VRAM_GiB",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Clone repo + install deps

Branch **`feature/cuda-torch-train`** has MCMC / antialiased / resume.

In [ ]:
import os
REPO = "/content/splats"
BRANCH = "feature/cuda-torch-train"
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch https://github.com/aidazsolt-bot/splats.git {REPO}
else:
    %cd {REPO}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
%cd {REPO}
!pip -q install -U pip
!pip -q install -r engines_cuda/requirements.txt
!python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA ok', _C is not None)"

## 3. Mount Drive and set dataset path

Expected layout:
```
MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt + images/
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/blackberry-splats/dataset"
# Separate out dir so resume settings stay MCMC-specific
OUT = "/content/drive/MyDrive/blackberry-splats/cuda_out_mcmc"

import os
assert os.path.isfile(f"{DATA}/cameras.txt"), f"Missing COLMAP at {DATA}"
assert os.path.isdir(f"{DATA}/images"), f"Missing images/ at {DATA}"
os.makedirs(OUT, exist_ok=True)
print("DATA", DATA)
print("OUT", OUT)

## 4. Smoke train (optional)

Short stack check. Uses `default` strategy + high `data-factor` to finish quickly.

In [ ]:
%cd /content/splats
!python engines_cuda/run_train.py --smoke --data "{DATA}" --out "{OUT}/smoke" \
  --data-factor 4 --strategy default

## 5. Full train (MCMC / Postshot-like candidate)

Recipe from local note / `engines_cuda/STRATEGY_PLAN.md`:
`--strategy mcmc --cap-max 200000 --max-steps 45000 --antialiased`.

| GPU | `DATA_FACTOR` | Notes |
|-----|---------------|-------|
| T4 (free) | `2` then try `1` | 200k + factor 1 may OOM |
| L4 / A100 | `1` | Closest to local note |

After disconnect: set `RESUME = True` (same flags + same `OUT`).

In [ ]:
%cd /content/splats
RESUME = False  # True after Colab disconnect / timeout
DATA_FACTOR = 1  # T4: use 2 if CUDA OOM
CAP_MAX = 200000
MAX_STEPS = 45000

cmd = (
    f'python engines_cuda/run_train.py --data "{DATA}" --out "{OUT}" '
    f'--strategy mcmc --cap-max {CAP_MAX} --max-steps {MAX_STEPS} '
    f'--data-factor {DATA_FACTOR} --antialiased'
)
if RESUME:
    cmd += " --resume"
print(cmd)
!{cmd}

## 6. List outputs / optional mesh from checkpoint

In [ ]:
!ls -lh "{OUT}"
# Optional mesh bake from a numbered ckpt:
# !python engines_cuda/export/mesh_from_ckpt.py --ckpt "{OUT}/ckpt_45000.pt" --out "{OUT}/mesh_from_splat.ply"

## Optional: PyTorch3D (meshes — not required for 3DGS)

Colab *can* install PyTorch3D; pin to the runtime torch/CUDA. Skip for Gaussian training.

In [ ]:
# Optional — skip for Gaussian training
# import torch
# print(torch.__version__, torch.version.cuda)
# !pip -q install --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt251/download.html
# import pytorch3d
# print("pytorch3d", pytorch3d.__version__)